# NB04 (V3_1) — QUICK satellite transfer pilot
Reuse the downloaded **V3 NB04** Sentinel-2 and Google annual-embedding GeoTIFFs with **V3_1 NB03 warm-start probability maps**. **No Earth Engine calls, no downloads and no neural-network training.** Primary validation: leave one entire site out (Baego, MOS, Yangambi, Yoko), evaluate predicted UAV-derived fractional cover in percentage points.

Targets: (1) **continuous mean probability**, area-averaged without threshold; (2) **hard canopy fraction**, fraction of UAV pixels where p≥0.5. The two are different and must not be confused. UAV targets are generated only from **valid RGB** pixels; require at least 90% valid UAV coverage per satellite cell. This is a fast transfer-screening experiment, not final ecological validation. Spatial autocorrelation and UAV label uncertainty remain relevant.


## 00 — USER SETTINGS (all switches here)

In [ ]:
# QUICK PILOT SETTINGS
SCALES_M = [30, 50]  # 10 m may be added after quick test
FAMILIES_TO_TEST = ["RAW_ANNUAL", "EMB_ANNUAL", "RAW_EMBEDDING", "S2_QUARTERLY"]
TARGETS_TO_TEST = ["cover_fraction_p05", "mean_probability"]
MAX_SAMPLES_PER_MOSAIC = 3000
MIN_VALID_UAV = 0.90
SEED = 20261022
RUN_PILOT = True


## 01 — Discover cached satellite features and final UAV maps

In [ ]:
%pip -q install rasterio pandas numpy scikit-learn matplotlib geopandas
import json, math, tempfile, gc
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from affine import Affine
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                       Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None
OLD=ROOT/'Outputs/V3/04_satellite_Musanga/features'
MAPS=ROOT/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
OUT=ROOT/'Outputs/V3_1/04_satellite_quick_warmstart';OUT.mkdir(parents=True,exist_ok=True)
assert OLD.is_dir() and MAPS.is_dir(),('Missing existing predictor directory or final maps',OLD,MAPS)
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
MOSAICS={
 'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],
 'Yoko':['Yoko_kvo_georef.tif']}
FAMILY_BASE=['RAW_ANNUAL','EMB_ANNUAL','S2_QUARTERLY']
def feat_path(site,fam):
    return OLD/fam/f'{site}_{fam}_{YEARS[site]}_10m.tif'
def map_path(name):
    return MAPS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
available={}
for site,names in MOSAICS.items():
    available[site]=[f for f in FAMILY_BASE if feat_path(site,f).is_file()]
    for name in names:assert map_path(name).is_file(),('Missing NB03 map',name)
print('Existing downloaded satellite predictors (NO downloads):')
display(pd.DataFrame([dict(site=site,feature_families=', '.join(v),maps=len(MOSAICS[site]))
                      for site,v in available.items()]))
assert all('RAW_ANNUAL' in x for x in available.values()),'Need annual S2 for every site'


## 02 — Derive satellite-cell UAV fractions and sample cached predictors
Each satellite cell's two targets are computed from the final native-resolution probability map. **No extra threshold** is applied to mean probability. For the hard fractional-cover target, p≥0.5 counts as Musanga. Mosaic overlaps are kept separate during sampling. Satellite arrays are aligned to each cached 10 m predictor grid before aggregation.


In [ ]:
def scale_grid(ds,scale):
    assert abs(ds.transform.a-10)<1e-5 and abs(ds.transform.e+10)<1e-5
    factor=scale/10.
    return (math.ceil(ds.height/factor),math.ceil(ds.width/factor),
            ds.transform*Affine.scale(factor,factor))
def warp_band(src,shape,transform,resampling=Resampling.average):
    dest=np.full(shape,np.nan,dtype='float32')
    reproject(source=src,destination=dest,dst_transform=transform,
              dst_crs=src.crs,src_nodata=src.nodata,dst_nodata=np.nan,
              resampling=resampling,num_threads=2)
    return dest
def native_binary_temp(source,tmp):
    # Only thresholded canopy fraction needs binary native UAV pixels.
    profile=source.profile.copy()
    for k in ('photometric','alpha','extrasamples','NBITS','colorinterp'):profile.pop(k,None)
    profile.update(driver='GTiff',count=1,dtype='uint8',nodata=255,
                   compress='deflate',tiled=True,blockxsize=256,blockysize=256)
    with rasterio.open(tmp,'w',**profile) as out:
        for _,win in source.block_windows(1):
            z=source.read(1,window=win)
            b=np.full(z.shape,255,dtype='uint8')
            ok=z!=65535
            b[ok]=(z[ok]>=5000).astype('uint8')
            out.write(b,1,window=win)
def read_scaled_features(site,scale):
    paths={fam:feat_path(site,fam) for fam in available[site]}
    refpath=paths['RAW_ANNUAL']
    with rasterio.open(refpath) as ref:
        h,w,tr=scale_grid(ref,scale)
        crs=ref.crs
    feats={}
    for fam,p in paths.items():
        with rasterio.open(p) as ds:
            assert ds.crs==crs and ds.nodata==-9999
            cube=np.stack([warp_band(rasterio.band(ds,i),(h,w),tr) for i in range(1,ds.count+1)])
            feats[fam]=cube
    return feats,(h,w,tr,crs)
def sample_site(site,scale):
    feats,(h,w,tr,crs)=read_scaled_features(site,scale)
    families={
        'RAW_ANNUAL':['RAW_ANNUAL'],
        'EMB_ANNUAL':['EMB_ANNUAL'],
        'S2_QUARTERLY':['S2_QUARTERLY'],
        'RAW_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}
    # Only shared feature families are evaluated across all held-out sites.
    sample_rows=[]
    for name in MOSAICS[site]:
        with rasterio.open(map_path(name)) as src:
            assert src.count==1 and src.nodata==65535 and src.crs==crs
            mean=warp_band(rasterio.band(src,1),(h,w),tr)/10000
            # Valid coverage: native map 1=valid, 0=invalid (separate from probability).
            with tempfile.TemporaryDirectory(prefix='nb04_') as td:
                bfile=Path(td)/'binary.tif'
                native_binary_temp(src,bfile)
                with rasterio.open(bfile) as binary:
                    frac=warp_band(rasterio.band(binary,1),(h,w),tr)
                    # Coverage 0..1 from a native validity raster.
                validfile=Path(td)/'valid.tif'
                profile=src.profile.copy()
                for k in ('photometric','alpha','extrasamples','NBITS','colorinterp'):profile.pop(k,None)
                profile.update(driver='GTiff',count=1,dtype='uint8',nodata=None,
                               compress='deflate',tiled=True,blockxsize=256,blockysize=256)
                with rasterio.open(validfile,'w',**profile) as out:
                    for _,win in src.block_windows(1):
                        out.write((src.read(1,window=win)!=65535).astype('uint8'),1,window=win)
                with rasterio.open(validfile) as vv:
                    coverage=warp_band(rasterio.band(vv,1),(h,w),tr)
            ok=(np.isfinite(mean)&np.isfinite(frac)&(coverage>=MIN_VALID_UAV)
                &(mean>=0)&(mean<=1)&(frac>=0)&(frac<=1))
            rr,cc=np.where(ok)
            if not len(rr):continue
            rng=np.random.default_rng(SEED+len(name)*17+scale)
            take=rng.choice(len(rr),size=min(MAX_SAMPLES_PER_MOSAIC,len(rr)),replace=False)
            rr=rr[take];cc=cc[take]
            frame=pd.DataFrame(dict(site=site,mosaic=name,row=rr,col=cc,
                mean_probability=mean[rr,cc],cover_fraction_p05=frac[rr,cc]))
            for fam,cube in feats.items():
                for bi in range(cube.shape[0]):
                    frame[f'{fam}__{bi:02d}']=cube[bi,rr,cc]
            sample_rows.append(frame)
            print('SAMPLED',site,name,scale,'m',len(frame),flush=True)
    if not sample_rows:return pd.DataFrame(),families
    return pd.concat(sample_rows,ignore_index=True),families


## 03 — Leave-one-site-out satellite prediction
Fixed gradient-boosting regressor; report **held-out-site** R², MAE, RMSE and bias in cover percentage points. Baseline predicts the training sites' mean; satellite models must beat it. This quick pilot does **not** involve hyperparameter tuning.


In [ ]:
assert RUN_PILOT
ALL=[]
for scale in SCALES_M:
    sample=[]
    for site in MOSAICS:
        x,_=sample_site(site,scale)
        assert len(x),('No samples',site,scale)
        sample.append(x)
    frame=pd.concat(sample,ignore_index=True)
    for family in FAMILIES_TO_TEST:
        parts={'RAW_ANNUAL':['RAW_ANNUAL'],'EMB_ANNUAL':['EMB_ANNUAL'],
               'S2_QUARTERLY':['S2_QUARTERLY'],'RAW_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}[family]
        if any(not all(part in available[site] for part in parts) for site in MOSAICS):
            print('SKIP missing cached predictors:',family);continue
        columns=[c for c in frame if any(c.startswith(part+'__') for part in parts)]
        for holdout in MOSAICS:
            train=frame[frame.site!=holdout].dropna(subset=columns)
            test=frame[frame.site==holdout].dropna(subset=columns)
            if len(train)<100 or len(test)<10:continue
            for target in TARGETS_TO_TEST:
                y=train[target].to_numpy()
                m=HistGradientBoostingRegressor(learning_rate=.06,max_iter=180,
                    max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                    early_stopping=False,random_state=SEED)
                m.fit(train[columns].to_numpy(),y)
                pred=np.clip(m.predict(test[columns].to_numpy()),0,1)
                truth=test[target].to_numpy()
                baseline=np.full(len(test),y.mean())
                for typ,pred0 in [('satellite',pred),('training_mean',baseline)]:
                    ALL.append(dict(scale_m=scale,family=family,heldout_site=holdout,
                        target=target,model=typ,n_test=len(truth),
                        R2=r2_score(truth,pred0),
                        MAE_pp=100*mean_absolute_error(truth,pred0),
                        RMSE_pp=100*np.sqrt(mean_squared_error(truth,pred0)),
                        bias_pp=100*np.mean(pred0-truth)))
    del frame,sample
    gc.collect()
result=pd.DataFrame(ALL)
assert not result.empty,'No complete held-out-site comparison available'
path=OUT/'04_quick_LOSO_satellite_verification.csv'
result.to_csv(path,index=False)
print('=== SATELLITE PREDICTION — HELD-OUT SITES (LOWER MAE IS BETTER) ===')
display(result[result.model=='satellite'].sort_values(['target','scale_m','family','heldout_site']).round(3))
print('=== MEAN MAE (PP) ACROSS FOUR HELD-OUT SITES ===')
display(result.groupby(['target','scale_m','family','model'],as_index=False).MAE_pp.mean().sort_values(['target','MAE_pp']).round(3))
print('Saved',path)
